# Accelerate Attention — Benchmark on NVIDIA L4

Compares three attention implementations exposed by `modules/attention.py`:

- `baseline` — original hand-written QKᵀ → softmax → V (no kernel fusion)
- `flash`    — PyTorch SDPA, which dispatches to the **FlashAttention-2** kernel on L4 (sm_89) with bf16/fp16
- `swa`      — Sliding Window Attention (Longformer-style). Uses `flash-attn`'s native `window_size` if the library is installed, otherwise falls back to a banded mask via SDPA (correct utility tradeoff, no compute speedup)

**Run on Google Colab with a single L4 GPU.**

Outputs:
1. Correctness check — baseline ≈ flash; SWA with full window ≈ baseline; SWA with small window diverges.
2. Speed sweep — forward / forward+backward latency vs seq_len ∈ {128, 256, 512, 1024, 2048, 4096}.
3. Speedup table and curves vs baseline.
4. Profiler kernel breakdown for one configuration.

## 1. Clone repository and switch to the `feat/AcceleAtt` branch

In [1]:
import os, subprocess

REPO_URL = 'https://github.com/Lynx-Zhang/DD2424-Project.git'
REPO_DIR = '/content/DD2424-Project'
BRANCH   = 'feat/AcceleAtt'

if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, REPO_DIR], check=True)
    
else:
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', BRANCH], check=True)

os.chdir(REPO_DIR)
print('cwd:', os.getcwd())
subprocess.run(['git', 'log', '-1', '--oneline'])

cwd: /content/DD2424-Project


CompletedProcess(args=['git', 'log', '-1', '--oneline'], returncode=0)

## 2. Install dependencies

`einops` is required by the model code. `flash-attn` is optional — if the install succeeds, SWA will use the native `window_size` argument (true O(N·w) compute); if it fails, SWA falls back to a banded mask via SDPA. The notebook works either way.

In [2]:
!pip install -q \
    tqdm==4.58.0 \
    requests==2.25.1 \
    importlib-metadata==3.7.0 \
    filelock==3.0.12 \
    tokenizers==0.20 \
    explainaboard_client==0.0.7 \
    einops==0.8.0 \
    transformers==4.46.3 \
    sacrebleu==2.5.1 \
    scikit-learn \
    matplotlib

# Optional: flash-attn enables true O(N*w) compute for SWA. If install fails,
# SWA automatically falls back to a banded mask via SDPA.
# !pip install -q flash-attn --no-build-isolation || echo '[WARN] flash-attn install failed; SWA will use dense-mask fallback.'

print("Dependencies installed.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 6.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 6.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.2/73.2 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 121.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 172.6 MB/s eta 0:00:000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.7/178.7 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.7/184.7 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

verify the version of pytorch

In [3]:
import torch
print('torch:', torch.__version__)
try:
    from torch.nn.attention.flex_attention import flex_attention
    print('✓ FlexAttention available — recommend using this for SWA')
except ImportError:
    print('✗ FlexAttention not available — torch too old')

torch: 2.10.0+cu128
✓ FlexAttention available — recommend using this for SWA


## 3. GPU sanity check

In [ ]:
import torch
print('torch                :', torch.__version__)
print('CUDA available       :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device               :', torch.cuda.get_device_name(0))
    print('compute capability   :', torch.cuda.get_device_capability(0))
    print('bf16 supported       :', torch.cuda.is_bf16_supported())

try:
    import flash_attn
    print('flash-attn version   :', flash_attn.__version__)
    HAS_FLASH_ATTN = True
except ImportError:
    print('flash-attn           : not installed (SWA will use dense-mask fallback)')
    HAS_FLASH_ATTN = False

## 4. Correctness check

All three modes are built with the **same weights**; we verify:
- `flash` numerical diff vs `baseline` is small (different kernel, same math, expect small fp error).
- `swa` with a window ≥ seq_len should also match `baseline` (the band covers the whole sequence).
- `swa` with a small window deviates — confirms the mask is doing something.

In [ ]:
import torch
from config import GPT2Config
from modules.attention import CausalSelfAttention

device = 'cuda'
torch.manual_seed(0)

cfg = GPT2Config()
baseline = CausalSelfAttention(cfg).to(device).eval()

def clone_layer(impl, swa_w=128):
    c = GPT2Config(attn_impl=impl, swa_window_size=swa_w)
    layer = CausalSelfAttention(c).to(device).eval()
    layer.load_state_dict(baseline.state_dict())
    return layer

flash      = clone_layer('flash')
swa_wide   = clone_layer('swa', swa_w=1024)   # window >= seq_len → ~full attention
swa_small  = clone_layer('swa', swa_w=8)

B, N = 2, 64
x = torch.randn(B, N, cfg.hidden_size, device=device)
mask = torch.zeros(B, 1, 1, N, device=device)

with torch.no_grad():
    o_base  = baseline(x, mask)
    o_flash = flash(x, mask)
    o_wide  = swa_wide(x, mask)
    o_small = swa_small(x, mask)

print(f'flash      vs baseline: max abs diff = {(o_base - o_flash ).abs().max().item():.3e}  (expect ~1e-5)')
print(f'swa(w=1024) vs baseline: max abs diff = {(o_base - o_wide  ).abs().max().item():.3e}  (expect ~1e-5)')
print(f'swa(w=8)    vs baseline: max abs diff = {(o_base - o_small ).abs().max().item():.3e}  (expect LARGE — different math)')

## 5. Speed micro-benchmark

Measures forward and forward+backward time of a single `CausalSelfAttention` layer (B=8, H=12, D=64 — GPT-2 small head config) in bf16. CUDA events for timing, 10 warmup iters, 30 measurement iters.

In [ ]:
import gc, torch
from config import GPT2Config
from modules.attention import CausalSelfAttention

def bench(impl, seq_len, batch=8, dtype=torch.bfloat16, n_warm=10, n_iter=30, swa_w=128):
    cfg = GPT2Config(attn_impl=impl, swa_window_size=swa_w)
    layer = CausalSelfAttention(cfg).to('cuda').to(dtype).train()

    x = torch.randn(batch, seq_len, cfg.hidden_size, device='cuda', dtype=dtype, requires_grad=True)
    mask = torch.zeros(batch, 1, 1, seq_len, device='cuda', dtype=dtype)
    g = torch.randn(batch, seq_len, cfg.hidden_size, device='cuda', dtype=dtype)

    # warmup
    for _ in range(n_warm):
        out = layer(x, mask)
        out.backward(g)
        x.grad = None
        for p in layer.parameters(): p.grad = None
    torch.cuda.synchronize()

    starter = torch.cuda.Event(enable_timing=True)
    ender   = torch.cuda.Event(enable_timing=True)

    # forward only
    starter.record()
    for _ in range(n_iter):
        with torch.no_grad():
            out = layer(x, mask)
    ender.record()
    torch.cuda.synchronize()
    fwd_ms = starter.elapsed_time(ender) / n_iter

    # forward + backward
    starter.record()
    for _ in range(n_iter):
        out = layer(x, mask)
        out.backward(g)
        x.grad = None
        for p in layer.parameters(): p.grad = None
    ender.record()
    torch.cuda.synchronize()
    fwd_bwd_ms = starter.elapsed_time(ender) / n_iter

    peak_mem_mb = torch.cuda.max_memory_allocated() / 1024**2
    torch.cuda.reset_peak_memory_stats()

    del layer, x, mask, g
    gc.collect(); torch.cuda.empty_cache()
    return fwd_ms, fwd_bwd_ms, peak_mem_mb

print('bench function ready')

In [ ]:
import json

seq_lens = [128, 256, 512, 1024, 2048, 4096]
configs = [
    ('baseline', None,  'baseline'),
    ('flash',    None,  'flash'),
    ('swa',      128,   'swa-w128'),
    ('swa',      256,   'swa-w256'),
]

results = {label: [] for _, _, label in configs}

for impl, w, label in configs:
    for N in seq_lens:
        try:
            fwd, fb, mem = bench(impl, N, swa_w=(w if w is not None else 128))
            results[label].append({'N': N, 'fwd_ms': fwd, 'fwd_bwd_ms': fb, 'peak_mem_mb': mem})
            print(f'{label:>10s}  N={N:>5d}  fwd={fwd:7.3f} ms  fwd+bwd={fb:7.3f} ms  peak={mem:7.1f} MB')
        except torch.cuda.OutOfMemoryError:
            results[label].append({'N': N, 'fwd_ms': float('nan'), 'fwd_bwd_ms': float('nan'), 'peak_mem_mb': float('nan')})
            print(f'{label:>10s}  N={N:>5d}  OOM')
            torch.cuda.empty_cache()

with open('bench_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print()
print('saved → bench_results.json')

## 6. Plot and speedup table

In [ ]:
import math
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for label, rows in results.items():
    Ns   = [r['N']           for r in rows]
    fwds = [r['fwd_ms']      for r in rows]
    fbs  = [r['fwd_bwd_ms']  for r in rows]
    mems = [r['peak_mem_mb'] for r in rows]
    axes[0].plot(Ns, fwds, marker='o', label=label)
    axes[1].plot(Ns, fbs,  marker='o', label=label)
    axes[2].plot(Ns, mems, marker='o', label=label)

for ax, title, ylabel in zip(
    axes,
    ['Forward latency', 'Forward + Backward latency', 'Peak GPU memory'],
    ['ms / call', 'ms / call', 'MB'],
):
    ax.set_xlabel('Sequence length')
    ax.set_ylabel(ylabel)
    ax.set_xscale('log', base=2)
    ax.set_yscale('log')
    ax.set_title(title)
    ax.grid(True, which='both', alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.savefig('bench_attention.png', dpi=140)
plt.show()
print('saved → bench_attention.png')

In [ ]:
# Speedup vs baseline (forward + backward)
labels = list(results.keys())
base_by_N = {r['N']: r['fwd_bwd_ms'] for r in results['baseline']}

header = f"{'N':>6s}" + ''.join(f"{l:>14s}" for l in labels)
print(header)
print('-' * len(header))
for N in seq_lens:
    row = f"{N:>6d}"
    for l in labels:
        d = {r['N']: r['fwd_bwd_ms'] for r in results[l]}
        v = d.get(N, float('nan'))
        b = base_by_N.get(N, float('nan'))
        if v != v or b != b:
            row += f"{'N/A':>14s}"
        else:
            row += f"{b / v:>13.2f}x"
    print(row)
print()
print('Values are speedup over baseline (>1 means faster).')

## 7. Profiler kernel breakdown

Confirms that `flash` fuses the attention into a single CUDA kernel, whereas `baseline` shows separate `matmul`, `softmax`, `dropout` calls. Run at one representative size.

In [ ]:
import torch, gc
from torch.profiler import profile, ProfilerActivity
from config import GPT2Config
from modules.attention import CausalSelfAttention

B, N = 8, 1024
dtype = torch.bfloat16

for impl, swa_w in [('baseline', 128), ('flash', 128), ('swa', 128)]:
    cfg = GPT2Config(attn_impl=impl, swa_window_size=swa_w)
    layer = CausalSelfAttention(cfg).to('cuda').to(dtype).train()
    x = torch.randn(B, N, 768, device='cuda', dtype=dtype, requires_grad=True)
    mask = torch.zeros(B, 1, 1, N, device='cuda', dtype=dtype)
    g = torch.randn_like(x)

    for _ in range(3):
        layer(x, mask).backward(g)
    torch.cuda.synchronize()

    with profile(activities=[ProfilerActivity.CUDA], record_shapes=False) as prof:
        for _ in range(5):
            out = layer(x, mask)
            out.backward(g)
    torch.cuda.synchronize()

    print('=' * 70)
    print(f'impl = {impl}   N={N}   B={B}   dtype=bf16')
    print('=' * 70)
    print(prof.key_averages().table(sort_by='cuda_time_total', row_limit=10))

    del layer, x, mask, g
    gc.collect(); torch.cuda.empty_cache()

## 8. What to put in the report

From the artifacts saved by this notebook:

- `bench_attention.png` — three-panel figure (fwd / fwd+bwd / peak memory). Use the **fwd+bwd** panel as the main speed figure.
- The speedup table from cell 6 — drop straight into the report.
- The profiler tables from cell 7 — screenshot the `baseline` vs `flash` tables side by side. `baseline` will show 4–6 separate CUDA kernels; `flash` will collapse into one `flash_attention_forward` / `efficient_attention_forward` entry.
- For the **utility** axis (perplexity / accuracy), train each `attn_impl` on WikiText-103 / Hyperpartisan / your sonnet dataset using the existing training scripts — switch by adding `attn_impl=...` to the `GPT2Config(...)` constructor in the relevant model file.

**Expected qualitative results on L4 bf16:**
- `flash` vs `baseline`: ~1.5× speedup at N=512, ~3× at N=2048, baseline OOMs at N=4096 while flash still runs.
- `swa` (with `flash-attn` installed): speedup grows with N/window ratio. At N=4096, w=128, expect ~5–8× over baseline. Without `flash-attn`, `swa` will be ~equal to baseline in speed but lower utility — useful for showing the math-only tradeoff.
- Profiler: `baseline` shows `aten::softmax` + multiple `aten::matmul` lines; `flash` consolidates into one fused kernel.